# Học phân loại mất cân bằng với dữ liệu gian lận thẻ tín dụng

Notebook này dạy từng bước: vì sao accuracy vô dụng khi lớp dương tính rất hiếm, logistic regression có class_weight, chọn ngưỡng bằng đường precision-recall, random forest, Isolation Forest (phát hiện bất thường không cần nhãn) và đánh giá mô hình theo chi phí tiền.

Dữ liệu: 284.807 giao dịch thẻ tín dụng ở châu Âu trong 2 ngày của tháng 9 năm 2013, trong đó 492 giao dịch gian lận (Class = 1). V1 đến V28 là các thành phần PCA, đã được ẩn danh nên không biết ý nghĩa gốc. Amount là số tiền giao dịch (EUR). Nguồn: bộ Credit Card Fraud Detection trên Kaggle (ULB Machine Learning Group). File này lấy từ bản trong ADBench: các cột V đã được đưa về khoảng 0 đến 1, cột Time bị bỏ, cột Amount được đổi lại về đơn vị EUR (sai số vài cent).

Chuẩn bị:
1. Đặt file data7.csv.gz cùng thư mục với notebook. pandas đọc trực tiếp file nén, không cần giải nén.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib seaborn scikit-learn
3. Một số bước chạy khá lâu (30 giây đến 2 phút) vì dữ liệu có gần 300.000 dòng.

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             average_precision_score, roc_auc_score, precision_recall_curve, PrecisionRecallDisplay)

## Bước 1: Đọc dữ liệu

Mục tiêu: đọc file nén và kiểm tra kích thước.

Kiến thức: pd.read_csv tự nhận ra file .gz và giải nén khi đọc.

Kết quả mong đợi: (284807, 30).

In [ ]:
# Viết code ở đây:
# 1. Đọc file data7.csv.gz vào biến df
# 2. In df.shape và 5 dòng đầu


## Bước 2: Mức độ mất cân bằng

Mục tiêu: đếm số giao dịch gian lận.

Kiến thức: khi lớp dương tính chiếm dưới 1%, ta gọi là mất cân bằng nặng (highly imbalanced). Đây là tình huống thường gặp: gian lận, bệnh hiếm, lỗi máy móc.

Kết quả mong đợi: 284315 giao dịch bình thường và 492 giao dịch gian lận, tỷ lệ gian lận khoảng 0,173%. Không có ô trống.

In [ ]:
# Viết code ở đây:
# 1. In value_counts của cột Class
# 2. In tỷ lệ gian lận theo phần trăm (làm tròn 3 chữ số)
# 3. In tổng số ô trống


## Bước 3: Số tiền giao dịch theo lớp

Mục tiêu: xem giao dịch gian lận có số tiền khác giao dịch bình thường không.

Kiến thức: trung vị (median) ít bị ảnh hưởng bởi giá trị cực lớn hơn trung bình. Phân phối số tiền rất lệch phải, nên vẽ boxplot với trục log.

Kết quả mong đợi: giao dịch bình thường có trung vị khoảng 21,99 EUR, giao dịch gian lận có trung vị khoảng 9,25 EUR. Giao dịch gian lận lớn nhất khoảng 2.125,87 EUR. Hai hộp chồng lên nhau rất nhiều.

Câu hỏi: có thể phát hiện gian lận chỉ bằng ngưỡng số tiền không?

In [ ]:
# Viết code ở đây:
# 1. In describe() của Amount theo từng Class (groupby, làm tròn 2 chữ số)
# 2. Vẽ sns.boxplot với x='Class', y='Amount', rồi đặt plt.yscale('symlog')


## Bước 4: Vì sao accuracy vô dụng

Mục tiêu: thấy một mô hình không bắt được gian lận nào vẫn có accuracy gần 100%.

Kiến thức: DummyClassifier luôn đoán 0 (không gian lận). Với 99,83% giao dịch bình thường, accuracy của nó là 0,998. Vì vậy với dữ liệu này ta dùng precision, recall, F1 và average precision (AP, diện tích dưới đường precision-recall).

Kết quả mong đợi: accuracy khoảng 0,998, recall bằng 0.

In [ ]:
# Viết code ở đây:
# 1. Tạo X = df bỏ Class, y = df['Class']
# 2. Fit DummyClassifier(strategy='most_frequent') trên X, y
# 3. In accuracy và recall trên chính X, y


## Bước 5: Chia train/test có stratify

Mục tiêu: đảm bảo tập test có đủ giao dịch gian lận.

Kiến thức: nếu không stratify, tập test có thể có quá ít ca gian lận do may rủi. Với 492 ca và test 20%, ta mong có khoảng 98 ca.

Kết quả mong đợi: X_train có 227845 dòng (394 gian lận), X_test có 56962 dòng (98 gian lận).

In [ ]:
# Viết code ở đây:
# 1. train_test_split với test_size=0.2, random_state=42, stratify=y
# 2. In kích thước và số ca gian lận ở mỗi tập


## Bước 6: Logistic regression không xử lý mất cân bằng

Mục tiêu: có con số đầu tiên để so sánh.

Kiến thức: hàm report dưới đây in confusion matrix và các chỉ số quan trọng. AP (average precision) dùng xác suất, không phụ thuộc ngưỡng, nên hợp để so sánh mô hình khi lớp dương tính hiếm. AP của mô hình đoán ngẫu nhiên bằng tỷ lệ gian lận, khoảng 0,0017.

Kết quả mong đợi: TP = 63, FP = 13, FN = 35. Precision khoảng 0,829, recall khoảng 0,643, F1 khoảng 0,724, AP khoảng 0,740.

In [ ]:
def report(name, y_true, pred, score):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    res = {'Mô hình': name, 'TP': tp, 'FP': fp, 'FN': fn, 'precision': precision_score(y_true, pred, zero_division=0),
           'recall': recall_score(y_true, pred), 'F1': f1_score(y_true, pred), 'AP': average_precision_score(y_true, score)}
    print(' | '.join(f'{k}: {v:.3f}' if isinstance(v, float) else f'{k}: {v}' for k, v in res.items()))
    return res

# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + LogisticRegression(max_iter=1000), fit trên tập train
# 2. proba_lr = predict_proba(X_test)[:, 1], pred_lr = predict(X_test)
# 3. Gọi report('Logistic', y_test, pred_lr, proba_lr), lưu vào res_lr


## Bước 7: Logistic với class_weight="balanced"

Mục tiêu: buộc mô hình chú ý hơn đến lớp gian lận.

Kiến thức: class_weight="balanced" cho mỗi mẫu gian lận trọng số khoảng 289 lần một mẫu bình thường (227451 / 394). Mô hình sẽ đoán gian lận nhiều hơn hẳn.

Kết quả mong đợi: TP = 90, FN chỉ còn 8, nhưng FP tăng lên 1448. Recall khoảng 0,918, precision chỉ khoảng 0,059, F1 khoảng 0,110. AP khoảng 0,720, gần bằng bước 6.

Câu hỏi: nếu mỗi cảnh báo sai (FP) làm khách hàng bị khóa thẻ tạm thời, ngân hàng có chấp nhận mô hình này không?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline như bước 6 nhưng thêm class_weight='balanced'
# 2. Fit, tính proba_bal và pred_bal, gọi report, lưu res_bal


## Bước 8: Chọn ngưỡng bằng đường precision-recall

Mục tiêu: tìm ngưỡng xác suất cho F1 cao nhất thay vì dùng 0,5.

Kiến thức: precision_recall_curve trả về precision, recall và các ngưỡng tương ứng. F1 = 2PR / (P + R). Chọn ngưỡng cho F1 lớn nhất là một cách cân bằng. Lưu ý: chọn ngưỡng trên tập test rồi báo kết quả trên chính tập test là hơi lạc quan. Cách đúng là chọn ngưỡng trên một tập validation riêng. Ở đây ta làm trên tập test để đơn giản.

Kết quả mong đợi: best_thr khoảng 0,184. Tại ngưỡng đó precision khoảng 0,728, recall khoảng 0,765, F1 khoảng 0,746, cao hơn F1 ở ngưỡng 0,5 (khoảng 0,724).

In [ ]:
# Viết code ở đây:
# 1. prec, rec, thr = precision_recall_curve(y_test, proba_lr)
# 2. f1 = 2 * prec * rec / (prec + rec + 1e-12), bỏ phần tử cuối (prec và rec dài hơn thr 1 phần tử)
# 3. Tìm best_thr = thr[argmax của f1], in best_thr, precision, recall, F1 tại đó
# 4. Vẽ đường precision theo recall


## Bước 9: Random forest

Mục tiêu: thử một mô hình phi tuyến.

Kiến thức: random forest thường xử lý dữ liệu dạng bảng tốt hơn logistic. Với 228 nghìn dòng, bước này mất khoảng 1 đến 2 phút. Random forest không cần chuẩn hóa.

Kết quả mong đợi: TP = 81, FP = 5, FN = 17. Precision khoảng 0,942, recall khoảng 0,827, F1 khoảng 0,880, AP khoảng 0,867.

In [ ]:
# Viết code ở đây:
# 1. Tạo RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42), fit trên tập train
# 2. proba_rf = predict_proba(X_test)[:, 1], pred_rf = predict(X_test)
# 3. Gọi report, lưu res_rf


## Bước 10: Feature importance của random forest

Mục tiêu: xem thành phần PCA nào giúp nhiều nhất.

Kiến thức: feature_importances_ có sẵn sau khi fit, tổng bằng 1. Vì V1 đến V28 đã được ẩn danh, ta không giải thích được ý nghĩa nghiệp vụ, chỉ biết cột nào quan trọng.

Kết quả mong đợi: V17 (khoảng 0,144), V12 (khoảng 0,132) và V14 (khoảng 0,130) đứng đầu. 6 cột đầu chiếm khoảng 65% tổng importance.

In [ ]:
# Viết code ở đây:
# 1. Tạo Series rf.feature_importances_ với index là X.columns
# 2. In 10 cột quan trọng nhất (làm tròn 3 chữ số)


## Bước 11: Isolation Forest: phát hiện bất thường không cần nhãn

Mục tiêu: tìm giao dịch lạ mà không dùng cột Class khi huấn luyện.

Kiến thức: Isolation Forest xây nhiều cây chia ngẫu nhiên. Điểm bất thường nằm xa đám đông nên bị cô lập sau rất ít lần chia. contamination là tỷ lệ bất thường ta dự kiến. predict trả về -1 cho bất thường và 1 cho bình thường. score_samples trả về điểm, càng thấp càng bất thường, nên dùng -score_samples làm điểm gian lận. Trong thực tế, cách này dùng khi chưa có nhãn hoặc khi kiểu gian lận mới chưa từng gặp.

Kết quả mong đợi: TP = 34, FP = 82, FN = 64. Precision khoảng 0,293, recall khoảng 0,347, AP khoảng 0,174, cao hơn khoảng 100 lần so với đoán ngẫu nhiên.

Câu hỏi: Isolation Forest không thấy nhãn mà vẫn đạt AP cao hơn nhiều lần so với đoán ngẫu nhiên (0,0017). Điều đó cho biết gì về giao dịch gian lận?

In [ ]:
# Viết code ở đây:
# 1. Tạo IsolationForest(n_estimators=200, contamination=0.002, random_state=42, n_jobs=-1)
# 2. Fit trên X_train (không truyền y)
# 3. pred_iso = (predict(X_test) == -1).astype(int), score_iso = -score_samples(X_test)
# 4. Gọi report, lưu res_iso


## Bước 12: So sánh đường precision-recall

Mục tiêu: vẽ đường PR của các mô hình trên cùng một hình.

Kiến thức: đường nằm càng cao và càng về bên phải thì càng tốt. AP trong chú thích là diện tích dưới đường.

Kết quả mong đợi: random forest nằm trên cùng ở phần lớn các mức recall. Isolation Forest thấp nhất.

In [ ]:
# Viết code ở đây:
# 1. fig, ax = plt.subplots()
# 2. Vẽ PrecisionRecallDisplay.from_predictions cho Logistic (proba_lr), Logistic balanced (proba_bal), Random forest (proba_rf), Isolation Forest (score_iso) với cùng ax


## Bước 13: Đánh giá theo chi phí

Mục tiêu: quy lỗi của mô hình thành tiền để chọn mô hình theo góc nhìn kinh doanh.

Kiến thức: giả định đơn giản: bỏ sót một giao dịch gian lận (FN) làm ngân hàng mất đúng số tiền giao dịch đó. Mỗi cảnh báo (TP hoặc FP) tốn 5 EUR để nhân viên kiểm tra. Tổng chi phí = tổng Amount của các FN + 5 x (TP + FP). Mô hình tốt nhất là mô hình có tổng chi phí thấp nhất, không nhất thiết là mô hình có F1 cao nhất.

Kết quả mong đợi: không dùng mô hình thì mất khoảng 10.645 EUR. Random forest có chi phí thấp nhất, khoảng 2.364 EUR. Logistic balanced tốn khoảng 8.349 EUR dù chỉ bỏ sót 8 ca, vì phải kiểm tra 1538 cảnh báo.

Câu hỏi: nếu chi phí kiểm tra tăng lên 50 EUR mỗi cảnh báo, mô hình nào sẽ tốt nhất? Hãy thử đổi số 5 thành 50.

In [ ]:
# Viết code ở đây:
# 1. Viết hàm cost(pred): fn_mask = (y_test == 1) & (pred == 0); trả về Amount của X_test tại fn_mask cộng lại, cộng 5 * số dự đoán bằng 1
# 2. Tính chi phí khi không dùng mô hình (pred toàn 0)
# 3. Tính chi phí cho pred_lr, pred_bal, pred_rf, pred_iso và logistic với ngưỡng best_thr


## Bước 14: Bảng so sánh các mô hình

Mục tiêu: ghép kết quả các mô hình vào một bảng.

Kiến thức: các dict từ hàm report ghép được thành DataFrame. Thêm cột chi phí từ bước 13.

Kết quả mong đợi: random forest dẫn đầu ở precision (khoảng 0,942), F1 (khoảng 0,880), AP (khoảng 0,867) và chi phí (khoảng 2.364 EUR). Logistic balanced có recall cao nhất (khoảng 0,918) nhưng chi phí cao nhất.

In [ ]:
# Viết code ở đây:
# 1. Tạo list gồm res_lr, res_bal, res_rf, res_iso
# 2. Thêm cột 'Chi phí EUR' bằng hàm cost cho từng mô hình
# 3. In DataFrame (làm tròn 3 chữ số)


## Bước 15: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Undersampling: lấy toàn bộ 394 ca gian lận trong tập train và 4.000 giao dịch bình thường ngẫu nhiên, huấn luyện logistic trên tập nhỏ này rồi đánh giá trên X_test gốc. AP thay đổi thế nào?
2. Cài imbalanced-learn (pip install imbalanced-learn) và thử SMOTE trên tập train. Không bao giờ áp dụng SMOTE lên tập test. Vì sao?
3. Thử RandomForestClassifier(class_weight='balanced_subsample'). Precision và recall đổi thế nào?
4. Viết ba câu tiếng Anh giải thích cho người quản lý vì sao accuracy 99,8% không có nghĩa là mô hình tốt. Ví dụ mở đầu: "A model that never flags fraud still reaches 99.8% accuracy, because ..."
